# WebGL Viewer in Jupyter Notebooks

Pycortex provides `cortex.webgl.jupyter` for embedding the interactive 3D
brain viewer directly in Jupyter notebook cells.

There are two display methods:

| Method | How it works | Best for |
|--------|-------------|----------|
| **IFrame** (default) | Starts a Tornado server and embeds it in an IFrame | Live sessions with full interactivity (WebSocket control, surface morphing) |
| **Static** | Generates a self-contained viewer directory served by a lightweight HTTP server | Sharing notebooks, documentation, quick previews |

## Display a single volume

The simplest way to show brain data is `cortex.webgl.jupyter.display()`.
Pass any pycortex data object (Volume, Vertex, Dataset) and it embeds
the viewer inline.

In a live Jupyter session you can use the default IFrame mode:

```python
viewer = cortex.webgl.jupyter.display(volume)
```

Or use the static method, which writes viewer files to a directory
and serves them via a lightweight HTTP server:

```python
viewer = cortex.webgl.jupyter.display(volume, method="static")
```

You can also pass ``output_dir`` to persist the viewer files to a
specific directory (useful for documentation or sharing):

In [ ]:
import cortex
import numpy as np

np.random.seed(1234)
volume = cortex.Volume.random(subject="S1", xfmname="fullhead")

viewer = cortex.webgl.jupyter.display(
    volume, method="static", output_dir="static_viewer"
)

Click and drag the brain to rotate, scroll to zoom. The toolbar on
the right lets you toggle overlays, adjust the colormap, and switch
surface types.

## Multiple datasets

Pass a ``cortex.Dataset`` to display multiple data layers with a
dropdown to switch between them:

```python
ds = cortex.Dataset(
    random_vol_1=volume,
    random_vol_2=cortex.Volume.random("S1", "fullhead"),
)
viewer = cortex.webgl.jupyter.display(ds)
```

## Context manager

Use the ``with`` statement to ensure cleanup happens automatically:

```python
with cortex.webgl.jupyter.display(volume, method="static") as viewer:
    pass  # viewer is displayed; do other work here
# Server and temp files are cleaned up on exit
```

## Programmatic control (IFrame mode)

In IFrame mode, you can control the viewer programmatically via
WebSocket. Always call ``get_client()`` in a **separate cell** from
``display()`` — it blocks until the browser connects.

```python
# Cell 1: start the viewer
server = cortex.webgl.jupyter.display(volume)
```

```python
# Cell 2: get a control handle
client = server.get_client()

# Rotate the view
client._set_view(azimuth=45, altitude=30)

# Switch to inflated surface (mix=1.0 is fully inflated)
client._set_view(mix=1.0)

# Capture a screenshot
client.getImage("screenshot.png")
```

## Closing viewers

Close a single viewer:

```python
viewer.close()
```

Or shut down all active static viewers at once:

```python
cortex.webgl.jupyter.close_all()
```

## Remote Jupyter setups

If running Jupyter on a remote server (JupyterHub, SSH tunnel,
cloud VM), set environment variables so IFrame URLs resolve correctly:

```bash
# For IFrame/Tornado mode:
export CORTEX_JUPYTER_IFRAME_HOST=my-server.example.com

# For static mode:
export CORTEX_JUPYTER_STATIC_HOST=0.0.0.0
```